# S5 Methods Across Ready Seeds

This notebook builds per-eta eval plots for the S5 method sweeps across the three training seeds you listed.

It compares:

- `Offline BC`
- `NAIL-forward, greedy rollout`
- `NAIL-reverse, greedy rollout`
- `NAIL-forward, sampled rollout`
- `TM OPD`

The notebook expects synced run artifact directories with at least:

- `eval_history.jsonl`
- `last_eval.json`
- `run_meta.json`

under one of these roots:

- the local repo tree itself
- `analysis/imports/s5_online_seed_sweeps/blocklab/`
- `analysis/imports/s5_online_seed_sweeps/aics/`
- `analysis/cache/s5_online_seed_sweeps/dev_node/`
- `analysis/cache/s5_online_seed_sweeps/aics/`

If duplicate `NAIL-reverse, greedy rollout` runs exist, the discovery helper now prefers corrected reruns with fixed-code markers.
You can also pin exact run directories with `analysis/cache/s5_online_seed_sweeps/run_overrides.json` using:

- `preferred_run_ids`: list of exact run directories to force-pick
- `excluded_run_ids`: list of exact run directories to ignore


In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Resolve the repo root whether the notebook is opened from the repo root or the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.plot_s5_online_seed_sweeps import (
    build_runs_df,
    coverage_table,
    default_output_dir,
    discover_s5_online_runs,
    plot_per_eta,
)

SEARCH_DIRS = [
    ROOT,
    ROOT / "analysis" / "imports" / "s5_online_seed_sweeps" / "blocklab",
    ROOT / "analysis" / "imports" / "s5_online_seed_sweeps" / "aics",
    ROOT / "analysis" / "cache" / "s5_online_seed_sweeps" / "dev_node",
    ROOT / "analysis" / "cache" / "s5_online_seed_sweeps" / "aics",
]

S5_M = 21
TEACHER_SEED = 20260417
SUBSET_SIZE = None  # Set this to an integer if you want strict filtering, e.g. 12_000_000.
SEEDS = [20260417, 20260418, 20260419]
ETAS = [0.0, 0.1, 0.7]

SAVE_PLOTS = True
PLOT_EXPORT_DIR = default_output_dir(ROOT, m=S5_M, teacher_seed=TEACHER_SEED)
PLOT_EXPORT_DIR


In [ ]:
# Discover matching S5 method runs from the local repo tree and synced import/cache directories.
records = discover_s5_online_runs(
    SEARCH_DIRS,
    m=S5_M,
    seeds=SEEDS,
    etas=ETAS,
    subset_size=SUBSET_SIZE,
    teacher_seed=TEACHER_SEED,
)

if not records:
    raise RuntimeError(
        "No matching S5 method runs were found. Sync run artifact directories into analysis/imports/s5_online_seed_sweeps/ or analysis/cache/s5_online_seed_sweeps/."
    )

runs_df, run_data = build_runs_df(records)
runs_df


In [ ]:
# Audit which NAIL-reverse runs were discovered and why they were selected.
runs_df.loc[
    runs_df["method"].eq("NAIL-reverse, greedy rollout"),
    [
        "eta",
        "seed",
        "source_kind",
        "reverse_variant",
        "selection_reason",
        "completed",
        "final_iter",
        "run_id",
    ],
].sort_values(["eta", "seed", "selection_rank", "final_iter"], ascending=[True, True, True, False])


In [ ]:
# Coverage summary so we can see which seed/method/eta combinations are already available.
coverage_df = coverage_table(
    runs_df,
    run_data,
    seeds=SEEDS,
    etas=ETAS,
)
coverage_df


In [ ]:
# Save one per-eta figure for clean_full_exact and also show it in the notebook.
plot_per_eta(
    run_data,
    runs_df,
    metric="val/clean_full_exact",
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)


In [ ]:
# Save one per-eta figure for clean_final_exact and also show it in the notebook.
plot_per_eta(
    run_data,
    runs_df,
    metric="val/clean_final_exact",
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)
